In [ ]:
!label-studio --version


In [3]:
!pip install label-studio


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/103.3 MB ? eta -:--:--
   ---------------------------------------- 1.0/103.3 MB 17.1 MB/s eta 0:00:06
    --------------------------------------- 1.3/103.3 MB 2.8 MB/s eta 0:00:38
   - -------------------------------------- 4.2/103.3 MB 8.0 MB/s eta 0:00:13
   -- ------------------------------------- 5.2/103.3 MB 6.2 MB/s eta 0:00:16
   -- ------------------------------------- 5.2/103.3 MB 6.2 MB/s eta 0:00:16
   -- ----------------------

  DEPRECATION: Building 'attr' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'attr'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'drf-flex-fields' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'drf-flex-fields'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'drf-generators' using the legacy setup.py bdist_w

   ---------------------------------------- 0.0/605.7 kB ? eta -:--:--
   --------------------------------------- 605.7/605.7 kB 12.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/14.6 MB ? eta -:--:--
   -------------------------- ------------- 9.7/14.6 MB 45.0 MB/s eta 0:00:01
   ---------------------------------------- 14.6/14.6 MB 35.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   --------------------------------- ------ 6.8/8.3 MB 33.3 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 29.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.7 MB ? eta -:--:--
   ---------------------------------------- 4.7/4.7 MB 35.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/948.6 kB ? eta -:--:--
   --------------------------------------- 948.6/948.6 kB 24.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/38.9 MB ? eta -:--:--
   --------- -------------------

In [ ]:
label studio

In [1]:
pip install PyMuPDF transformers datasets streamlit torch pandas scikit-learn

   ---------------------------------------- 0.0/18.4 MB ? eta -:--:--
   -- ------------------------------------- 1.3/18.4 MB 9.3 MB/s eta 0:00:02
   ------------- -------------------------- 6.3/18.4 MB 17.6 MB/s eta 0:00:01
   ------------------------ --------------- 11.3/18.4 MB 19.7 MB/s eta 0:00:01
   ---------------------------------------  18.4/18.4 MB 24.0 MB/s eta 0:00:01
   ---------------------------------------- 18.4/18.4 MB 22.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   --------------------------- ------------ 8.4/12.0 MB 39.4 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 33.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/566.1 kB ? eta -:--:--
   ---------------------------------------- 566.1/566.1 kB 6.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 23.5 MB/s eta 0:00:00
   -----------

In [2]:
"""
This module splits your big PDF into individual contracts and then into clauses.
Like opening 25 candy boxes and breaking each chocolate bar into squares.
"""

import fitz  # PyMuPDF for PDF processing
import re
from typing import List, Dict

def split_combined_pdf(input_path: str, output_folder: str):
    """
    Splits your 25-in-1 PDF into separate files.
    If each contract starts with "SAAS AGREEMENT", we split there.
    """
    doc = fitz.open(input_path)
    contract_boundaries = []
    
    # Find where each contract starts
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()
        
        # LOOK HERE: Change this pattern based on your PDF structure
        if "SAAS AGREEMENT" in text.upper():
            contract_boundaries.append(page_num)
    
    # Split and save
    for i in range(len(contract_boundaries)):
        start = contract_boundaries[i]
        end = contract_boundaries[i+1] if i+1 < len(contract_boundaries) else len(doc)
        
        new_doc = fitz.open()
        new_doc.insert_pdf(doc, from_page=start, to_page=end-1)
        new_doc.save(f"{output_folder}/contract_{i+1}.pdf")
        new_doc.close()
    
    print(f"✅ Split into {len(contract_boundaries)} contracts")

def extract_clauses_from_pdf(pdf_path: str) -> List[Dict]:
    """
    Breaks a contract into clauses based on numbering like "1.1", "2.3", etc.
    Returns a list: [{"clause_number": "1.1", "text": "The customer shall..."}]
    """
    doc = fitz.open(pdf_path)
    full_text = ""
    
    # Extract all text from PDF
    for page in doc:
        full_text += page.get_text() + "\n"
    
    # LOOK HERE: This regex finds clauses like "1.1", "2.3.1", "12.4"
    clause_pattern = r'(\d+\.\d+(\.\d+)?)\s+([A-Z][^.!?]*[.!?])'
    matches = re.finditer(clause_pattern, full_text, re.MULTILINE)
    
    clauses = []
    for match in matches:
        clauses.append({
            "clause_number": match.group(1),
            "text": match.group(3).strip(),
            "page": 0  # You can enhance this to track actual pages
        })
    
    return clauses

# Example usage in Jupyter:
# clauses = extract_clauses_from_pdf("contracts/contract_1.pdf")
# print(f"Found {len(clauses)} clauses")

In [3]:
python
"""
This is where lawyers put colored stickers on each clause.
You can do this manually in Excel or semi-automatically with keywords.
"""

import pandas as pd

def create_labeled_dataset(clauses: List[Dict]) -> pd.DataFrame:
    """
    Converts clauses into a table for labeling. Export to Excel, let lawyers fill 'risk_level'.
    risk_level: 0=Safe, 1=Medium, 2=High
    """
    df = pd.DataFrame(clauses)
    df['risk_level'] = -1  # Unlabeled
    
    # LOOK HERE: Auto-label some obvious clauses to save lawyer time
    high_risk_keywords = ['unlimited liability', 'indemnification', 'warranty', 'breach']
    medium_risk_keywords = ['termination', 'payment', 'confidentiality']
    
    for idx, row in df.iterrows():
        text_lower = row['text'].lower()
        if any(kw in text_lower for kw in high_risk_keywords):
            df.at[idx, 'risk_level'] = 2
        elif any(kw in text_lower for kw in medium_risk_keywords):
            df.at[idx, 'risk_level'] = 1
        elif len(text_lower) > 20:  # If it's a decent length sentence
            df.at[idx, 'risk_level'] = 0
    
    return df

# Save to Excel, send to lawyer
# df = create_labeled_dataset(all_clauses)
# df.to_excel("labeling_task.xlsx", index=False)

NameError: name 'python' is not defined

In [ ]:
python
"""
This is the robot's brain training school.
We use a BERT model that already understands English,
and teach it the special language of contracts.
"""

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from datasets import Dataset
import pandas as pd

class ContractRiskModel:
    def __init__(self, model_name: str = "nlpaueb/legal-bert-base-uncased"):
        """
        Initialize with Legal-BERT, a version of BERT trained on legal documents.
        This is like giving the robot a law school degree before training it further.
        """
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=3,  # 0: Safe, 1: Medium, 2: High
            id2label={0: "Safe", 1: "Medium", 2: "High"},
            label2id={"Safe": 0, "Medium": 1, "High": 2}
        )
    
    def prepare_data(self, df: pd.DataFrame):
        """
        Convert our labeled clauses into a format BERT can eat.
        BERT digests text differently - it needs special tokens and padding.
        """
        # Remove unlabeled rows
        df = df[df['risk_level'] != -1]
        
        # Split into train and test
        train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['risk_level'])
        
        # Convert to Hugging Face Dataset
        train_dataset = Dataset.from_pandas(train_df)
        test_dataset = Dataset.from_pandas(test_df)
        
        # Tokenize function
        def tokenize_function(examples):
            return self.tokenizer(
                examples['text'],
                padding="max_length",
                truncation=True,
                max_length=128
            )
        
        # Apply tokenization
        train_dataset = train_dataset.map(tokenize_function, batched=True)
        test_dataset = test_dataset.map(tokenize_function, batched=True)
        
        # Format for PyTorch
        train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'risk_level'])
        test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'risk_level'])
        
        return train_dataset, test_dataset
    
    def train(self, train_dataset, test_dataset, output_dir: str = "./risk_model"):
        """
        The actual training loop. This is where the robot studies day and night.
        """
        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=3,        # How many times to read the textbook
            per_device_train_batch_size=8,  # Read 8 clauses at a time
            per_device_eval_batch_size=8,
            warmup_steps=100,          # Start slow to avoid confusion
            weight_decay=0.01,         # Prevent overfitting (robot memorizing instead of learning)
            logging_dir="./logs",
            logging_steps=10,
            evaluation_strategy="epoch",  # Test after each textbook round
            save_strategy="epoch",
            load_best_model_at_end=True,
        )
        
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
        )
        
        trainer.train()
        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)
        
        print(f"✅ Model trained and saved to {output_dir}")

# Example usage:
# model = ContractRiskModel()
# train_ds, test_ds = model.prepare_data(labeled_df)
# model.train(train_ds, test_ds)

In [4]:
python
"""
The robot's final exam: Read new contracts and color the clauses.
"""

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class RiskPredictor:
    def __init__(self, model_path: str = "./risk_model"):
        """
        Load the trained robot brain.
        """
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.model.eval()  # Set to evaluation mode
    
    def predict_clause(self, clause_text: str) -> dict:
        """
        Takes one clause, returns risk prediction with confidence score.
        """
        # Tokenize the clause
        inputs = self.tokenizer(
            clause_text,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        
        # Predict without updating gradients (faster)
        with torch.no_grad():
            outputs = self.model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        
        # Get the most likely class and its confidence
        confidence, predicted_class = torch.max(predictions, dim=1)
        
        risk_labels = {0: "Safe", 1: "Medium", 2: "High"}
        
        return {
            "risk_level": risk_labels[predicted_class.item()],
            "confidence": confidence.item(),
            "clause_text": clause_text
        }
    
    def analyze_contract(self, clauses: List[Dict]) -> List[Dict]:
        """
        Analyzes a whole contract clause by clause.
        """
        results = []
        for clause in clauses:
            prediction = self.predict_clause(clause['text'])
            results.append({
                "clause_number": clause['clause_number'],
                "risk_level": prediction['risk_level'],
                "confidence": prediction['confidence'],
                "text": clause['text']
            })
        return results

# Example usage:
# predictor = RiskPredictor()
# results = predictor.analyze_contract(clauses_from_new_contract)

NameError: name 'python' is not defined